# Word Embeddings

## Overview

Word embeddings represent words as dense vectors in a continuous space where semantically similar words are geometrically close. They capture meaning in a way sparse bag-of-words vectors cannot.

**Comparison:**

| Representation | Dimensions | Captures meaning | Handles unseen words |
|---|---|---|---|
| One-hot / BoW | V (vocab size) | No | No |
| TF-IDF | V | Partly (co-occurrence) | No |
| Word2Vec | 50–300 | Yes | No |
| GloVe | 50–300 | Yes | No |
| fastText | 50–300 | Yes | Yes (subword) |
| Contextual (BERT) | 768+ | Yes (context-aware) | Yes |

**Classic embedding properties:**
- king − man + woman ≈ queen
- Similar words cluster together in embedding space
- Dimensionality much lower than vocabulary size

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE

# Simulate documents for training a Word2Vec model
# Ecological monitoring report sentences
sentences = [
    ["nitrate", "levels", "elevated", "site", "north"],
    ["phosphorus", "concentration", "high", "upstream"],
    ["species", "richness", "increased", "restoration"],
    ["water", "quality", "improved", "buffer", "strips"],
    ["macroinvertebrate", "taxa", "returned", "restored", "reach"],
    ["nitrate", "reduction", "wetland", "treatment"],
    ["riparian", "vegetation", "recovery", "observed"],
    ["dissolved", "oxygen", "levels", "healthy", "range"],
    ["fish", "passage", "restored", "culvert", "replacement"],
    ["biodiversity", "indices", "positive", "trend"],
    ["turbidity", "spike", "rainfall", "event"],
    ["algal", "bloom", "upstream", "sampling"],
    ["temperature", "threshold", "salmonids", "exceeded"],
    ["biological", "oxygen", "demand", "downstream"],
    ["catchment", "restoration", "nitrogen", "loading", "reduced"],
    ["wetland", "phosphorus", "buffer", "constructed"],
    ["macroinvertebrate", "index", "declined", "quarter"],
    ["water", "temperature", "stable", "monitoring"],
    ["species", "richness", "taxa", "diversity", "recovery"],
    ["nitrate", "phosphorus", "nutrients", "elevated", "concern"],
]
print(f"{len(sentences)} training sentences")
print(f"Vocab preview: {sorted(set(w for s in sentences for w in s))[:15]}")

---
## Training Word2Vec

In [ ]:
try:
    from gensim.models import Word2Vec
    model = Word2Vec(
        sentences=sentences,
        vector_size=50,    # embedding dimensions
        window=3,          # context window size
        min_count=1,       # min word frequency
        workers=1,
        sg=1,              # 1=skip-gram, 0=CBOW
        epochs=200,
        seed=42,
    )
    print(f"Vocabulary size: {len(model.wv)}")
    print(f"Embedding dimensions: {model.vector_size}")
    print(f"\nVector for 'nitrate' (first 10 dims):")
    print(model.wv['nitrate'][:10].round(4))
    print(f"\nMost similar to 'restoration':")
    for word, sim in model.wv.most_similar('restoration', topn=5):
        print(f"  {word:20s}: cosine={sim:.3f}")
    print(f"\nMost similar to 'nitrate':")
    for word, sim in model.wv.most_similar('nitrate', topn=5):
        print(f"  {word:20s}: cosine={sim:.3f}")
except ImportError:
    print("pip install gensim")
    print("Word2Vec('sentences', vector_size=100, window=5, min_count=1)")
    print("Trained on unlabelled text corpus")
    print("Captures semantic similarity: similar(nitrate, phosphorus) > similar(nitrate, fish)")

---
## Visualising Embedding Space

In [ ]:
try:
    from gensim.models import Word2Vec
    model = Word2Vec(sentences=sentences, vector_size=50, window=3,
                     min_count=1, workers=1, sg=1, epochs=200, seed=42)
    words  = list(model.wv.key_to_index.keys())
    vecs   = np.array([model.wv[w] for w in words])
    # PCA to 2D
    pca = PCA(n_components=2, random_state=42)
    vecs_2d = pca.fit_transform(vecs)
    # Colour by semantic group
    water_quality = {'nitrate','phosphorus','turbidity','oxygen','temperature'}
    biology       = {'species','richness','taxa','macroinvertebrate','biodiversity','fish'}
    restoration   = {'restoration','restored','recovery','buffer','wetland','riparian'}
    def get_color(word):
        if word in water_quality: return '#e74c3c'
        if word in biology:       return 'steelblue'
        if word in restoration:   return '#2ecc71'
        return 'grey'
    fig, ax = plt.subplots(figsize=(10,7))
    for i, word in enumerate(words):
        ax.scatter(vecs_2d[i,0], vecs_2d[i,1],
                   color=get_color(word), s=60, alpha=0.8, zorder=3)
        ax.annotate(word, (vecs_2d[i,0]+0.02, vecs_2d[i,1]+0.02), fontsize=8)
    from matplotlib.patches import Patch
    legend = [Patch(color='#e74c3c',label='Water quality'),
              Patch(color='steelblue',label='Biology'),
              Patch(color='#2ecc71',label='Restoration'),
              Patch(color='grey',label='Other')]
    ax.legend(handles=legend)
    ax.set_title('Word2Vec Embeddings (PCA to 2D): Semantic Clusters')
    plt.tight_layout(); plt.show()
except ImportError:
    print("pip install gensim")

---
## Document Embeddings via Averaging

In [ ]:
try:
    from gensim.models import Word2Vec
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score, StratifiedKFold
    model = Word2Vec(sentences=sentences, vector_size=50, window=3,
                     min_count=1, workers=1, sg=1, epochs=200, seed=42)
    # Simple document embedding: average word vectors
    def doc_embedding(tokens, wv, vector_size=50):
        vecs = [wv[t] for t in tokens if t in wv]
        return np.mean(vecs, axis=0) if vecs else np.zeros(vector_size)
    # Extend corpus for classification demo
    import re
    rng = np.random.default_rng(42)
    concern_sents = [
        "nitrate levels elevated site concern",
        "phosphorus concentration high upstream problem",
        "turbidity spike rainfall event flagged",
        "macroinvertebrate index declined quarter",
        "algal bloom upstream sampling concern",
        "temperature threshold exceeded critical",
        "biological oxygen demand downstream elevated",
        "water quality deteriorated monitoring period",
    ]
    positive_sents = [
        "species richness increased restoration success",
        "water quality improved buffer strips installed",
        "macroinvertebrate taxa returned restored reach",
        "biodiversity indices positive trend recovery",
        "riparian vegetation recovery observed buffer",
        "dissolved oxygen levels healthy range restored",
        "fish passage restored culvert replacement project",
        "wetland phosphorus reduction achieved success",
    ]
    docs = concern_sents + positive_sents
    y_docs = np.array([0]*8 + [1]*8)
    tokenised_docs = [d.split() for d in docs]
    X_embed = np.array([doc_embedding(tok, model.wv) for tok in tokenised_docs])
    cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
    scores = cross_val_score(LogisticRegression(max_iter=500, random_state=42),
                              X_embed, y_docs, cv=cv, scoring='f1')
    print(f"Doc embedding CV F1: {scores.mean():.3f} +/- {scores.std():.3f}")
    print("\nNote: with only 16 docs this demo is illustrative, not a real benchmark")
    print("Word embedding features improve with larger training corpora")
except ImportError:
    print("pip install gensim")

In [ ]:
# Pre-trained embeddings: using GloVe/fastText via gensim downloader
try:
    import gensim.downloader as api
    print("Available pre-trained models (subset):")
    info = api.info()
    for name in list(info['models'].keys())[:8]:
        m = info['models'][name]
        print(f"  {name:35s}: {m.get('parameters',{}).get('dimension','?')}d, "
              f"{m.get('file_size',0)//1e6:.0f}MB")
    print("\nTo load: model = api.load('glove-wiki-gigaword-50')")
    print("Then:     model.most_similar('restoration')")
    print("\nPre-trained on Wikipedia / Common Crawl -- much richer than")
    print("small domain corpora. Fine-tune with domain text when possible.")
except ImportError:
    print("pip install gensim")
    print("Pre-trained models: glove-wiki-gigaword-50/100/200/300")
    print("                    fasttext-wiki-news-subwords-300")
    print("                    word2vec-google-news-300")
    print("Load with: gensim.downloader.load('model-name')")

---

## Common Pitfalls

**1. Training Word2Vec on a corpus that is too small**  
Word2Vec requires substantial co-occurrence statistics to produce meaningful embeddings — typically tens of thousands of sentences. With fewer than a few thousand sentences, embeddings are noisy and unreliable. Use pre-trained embeddings (GloVe, fastText) and fine-tune rather than training from scratch on small corpora.

**2. Averaging word vectors without handling out-of-vocabulary words**  
If a document contains words not in the vocabulary, naive averaging skips them silently. For short documents, a single OOV word may represent 20-50% of the content. Always check OOV rate for your inference documents; use fastText (subword embeddings) to handle unseen words.

**3. Using static word embeddings for polysemous words**  
Word2Vec assigns one vector per word regardless of context: "bank" (financial) and "bank" (river) share the same embedding. For tasks where word sense matters, use contextual embeddings (BERT, RoBERTa) which produce different vectors for the same word in different contexts.

**4. Comparing cosine similarities from embeddings trained on different corpora**  
Cosine similarity is only meaningful within the same embedding space. Embeddings trained on general web text and domain-specific text are not directly comparable, and words that are similar in one space may not be similar in another.

**5. Forgetting to normalise vectors before computing cosine similarity manually**  
`np.dot(a, b)` computes the dot product, not cosine similarity, unless vectors are unit-normalised. Use `sklearn.metrics.pairwise.cosine_similarity()` or normalise with `a / np.linalg.norm(a)` before computing dot products.
---
*python_methods_library - Samantha McGarrigle*